# 양쪽 실행 빈 결과 판정 점검

합성 입력을 만들고 `empty_review`의 준비와 집계를 확인합니다. 실제 경로는 아래 변수에서 지정합니다.

In [ ]:
from pathlib import Path
import json
import tempfile

WORK = Path(tempfile.mkdtemp())
RESULT_A = WORK / "run_a"
RESULT_B = WORK / "run_b"
CHUNK_DIR = WORK / "chunks"
OUTPUT_DIR = WORK / "review"
for path in (RESULT_A, RESULT_B, CHUNK_DIR): path.mkdir()


## 합성 입력 예시

In [ ]:
chunk = {"chunk_id": "synthetic-001", "text": "합성 혜택 5% 할인", "page_no": 1}
(CHUNK_DIR / "sample.chunks.json").write_text(json.dumps({"chunks": [chunk]}, ensure_ascii=False), encoding="utf-8")
for folder in (RESULT_A, RESULT_B):
    (folder / "sample.result.json").write_text(json.dumps({"chunk_id": chunk["chunk_id"], "page_no": 1, "extraction": {"benefits": []}}, ensure_ascii=False), encoding="utf-8")


## prepare

In [ ]:
from payment_card.file_pipeline.extraction.empty_review import prepare

prepared = prepare(RESULT_A, RESULT_B, CHUNK_DIR, OUTPUT_DIR, top=8)
prepared["selected"]


## tally

In [ ]:
sheet = Path(prepared["sheet"])
sheet.write_text(sheet.read_text(encoding="utf-8").replace("[ ] 정상", "[x] 정상", 1), encoding="utf-8")
from payment_card.file_pipeline.extraction.empty_review import tally
tally(sheet)
